In [1]:
import json
import re

def debug_graph(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    nodes = {n['id']: n for n in data['nodes']}
    # Tạo map edges: source -> list of targets
    adj = {}
    for e in data['edges']:
        adj.setdefault(e['source'], []).append(e['target'])

    print(f"--- DEBUG REPORT: {file_path} ---")

    # 1. TRACE TỪ STATE_0
    root_id = data.get('root_id', 'state_0')
    print(f"\n[1] Root Node: {root_id}")
    root_actions = adj.get(root_id, [])
    for aid in root_actions:
        a_node = nodes[aid]
        print(f"    -> Action: {aid} | Type: {a_node.get('action_type')} | Status: {a_node['status']} | r_env: {a_node['metrics'].get('r_env')}")

    # 2. PHÂN TÍCH CHI TIẾT ACTION_444
    target_aid = 'action_444'
    if target_aid in nodes:
        print(f"\n[2] Deep Dive: {target_aid}")
        node = nodes[target_aid]
        code = node.get('extracted_lean_code', '')
        print(f"    - Status: {node['status']}")
        print(f"    - r_env: {node['metrics'].get('r_env')}")
        print(f"    - Subgoals (Children): {adj.get(target_aid, [])}")
        print(f"    - Extracted Code:\n---")
        print(code)
        print("---")
        
        # Kiểm tra từ khóa Lean 3 'in' thay vì '∈'
        if ' in ' in code:
            print(f"    ⚠️ CẢNH BÁO CÚ PHÁP: Tìm thấy từ khóa 'in' (Lean 3).")
            print(f"    Tại sao r_env vẫn bằng 1? Kiểm tra log verify...")
    else:
        print(f"\n[2] {target_aid} không tồn tại trong file này.")

    # 3. TRUY VẾT LỖI STITCH DẪN ĐẾN SAI LÊN TRÊN
    print("\n[3] Trace Stitching (Bottom-up failure):")
    # Tìm các action SOLVED nhưng có con là None hoặc con FAILED
    for nid, node in nodes.items():
        if node['type'] == 'AND' and node['status'] == 'SOLVED':
            children = adj.get(nid, [])
            # Nếu là skeleton mà không có con, hoặc có con nhưng con chưa SOLVED
            if node.get('action_type') == 'skeleton':
                unsolved_children = [cid for cid in children if nodes[cid]['status'] != 'SOLVED']
                if not children:
                    print(f"    ❌ LỖI LOGIC: {nid} là Skeleton SOLVED nhưng KHÔNG có subgoals.")
                elif unsolved_children:
                    print(f"    ❌ LỖI LOGIC: {nid} là Skeleton SOLVED nhưng có con chưa giải xong: {unsolved_children}")

    # 4. MÔ PHỎNG STITCH TẠI STATE_0
    print("\n[4] Simulate Final Stitch for state_0:")
    def get_proof(sid, visited):
        if sid in visited: return "--- LOOP ---"
        visited.add(sid)
        state_node = nodes[sid]
        # Tìm action SOLVED
        for aid in adj.get(sid, []):
            if nodes[aid]['status'] == 'SOLVED':
                a_node = nodes[aid]
                if a_node['action_type'] == 'tactic':
                    return a_node.get('extracted_lean_code', 'sorry')
                else:
                    # Skeleton
                    skel_code = a_node.get('extracted_lean_code', '')
                    child_ids = adj.get(aid, [])
                    # Đây là chỗ dễ sai nhất: Thứ tự con trong edges vs thứ tự sorry trong code
                    child_proofs = [get_proof(cid, visited.copy()) for cid in child_ids]
                    return f"Stitched({aid}) with {len(child_proofs)} children"
        return "sorry"

    print(f"    Result for state_0: {get_proof(root_id, set())}")

debug_graph('outputs/rollouts/deepseekv4/aime_1994_p3.json')

--- DEBUG REPORT: outputs/rollouts/deepseekv4/aime_1994_p3.json ---

[1] Root Node: state_0
    -> Action: action_0 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_1 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_2 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_3 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_4 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_5 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_6 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_7 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_8 | Type: tactic | Status: FAILED | r_env: 0.6153846153846154
    -> Action: action_9 | Type: tactic | Status: FAILED | r_env: 0.0
    -> Action: action_10 | Type: tactic | Status: FAILED | r_env: 0.9166666666666666
    -> Action: action_11 | Type: tactic | Status: FAILED | r_env: 0.9777777777777777
    -> Action: action_12 | Type: 